# ESM-DMS real-data analysis

This notebook runs the real-data ESM-DMS workflow through the current `esmDMS` class structure for cellular datasets `TpoR`, `Ube4b`, and `BRCA1`, and viral datasets `BF520` and `BG505`.

Notebook code is limited to configuration and orchestration around `esmDMS` methods. Missing class-level affordances are marked with `# TODO(esmDMS.py)`.


In [1]:
from pathlib import Path

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from esmDMS import CellularDMSInput, ViralDMSInput, ESMDMSConfig, esmDMS

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break

DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_data"
ANALYSIS_DIR = DATA_DIR / "esm_data_analysis"
SEQUENCE_DIR = ANALYSIS_DIR / "sequence_data"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid")
REPO_ROOT


/net/dali/home/barton/dhw28/popDMS/esmDMS/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/net/dali/home/barton/dhw28/popDMS/esmDMS')

## Dataset Registry

Cellular datasets use MaveDB nucleotide-count files. Viral datasets use paired mutant-DNA and mutant-virus codon-count files.


In [3]:
CELLULAR_DATASETS = {
    "TpoR": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "TpoR_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "TpoR_nucleotide_counts.csv",
    ),
    "Ube4b": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "Ube4b_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "Ube4b_nucleotide_counts.csv",
    ),
    "BRCA1": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "BRCA1_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "BRCA1_nucleotide_counts.csv",
    ),
}

VIRAL_DATASETS = {
    "BF520": ViralDMSInput(
        pre_files=tuple(RAW_DIR / f"BF520_mutDNA-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
        post_files=tuple(RAW_DIR / f"BF520_mutvirus-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
    ),
    "BG505": ViralDMSInput(
        pre_files=tuple(RAW_DIR / f"BG505_mutDNA-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
        post_files=tuple(RAW_DIR / f"BG505_mutvirus-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
    ),
}

DATASETS = {**CELLULAR_DATASETS, **VIRAL_DATASETS}
DATASET_KIND = {
    **{dataset: "cellular" for dataset in CELLULAR_DATASETS},
    **{dataset: "viral" for dataset in VIRAL_DATASETS},
}

pd.DataFrame(
    {"dataset": dataset, "kind": DATASET_KIND[dataset], "save_dir": str(SEQUENCE_DIR / dataset)}
    for dataset in DATASETS
)


,dataset,kind,save_dir
0,TpoR,cellular,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...
1,Ube4b,cellular,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...
2,BRCA1,cellular,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...
3,BF520,viral,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...
4,BG505,viral,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...


## Controls

The controls below configure Slurm job creation and submission. `SUBMIT_JOBS = True` matches the walkthrough pattern and submits jobs when the submission cells are run. `EMBEDDING_MAX_ACTIVE_JOBS` limits active Slurm array tasks per dataset.


In [4]:
# TODO(esmDMS.py): add a public available_layers() method so notebooks do not hard-code model layers.
LAYERS = list(range(34))
REPRESENTATIVE_LAYER = 33

ABSTRACTION_METHOD = "Embeddings"
ABSTRACTION_PARAMS = {"norm_scheme": "none"}
NORM_SCHEME = ABSTRACTION_PARAMS["norm_scheme"]

SUBMIT_JOBS = True
CLUSTER_PARTITION = "any_cpu"
SCRATCH_ROOT = "/scr"
PYTHON_EXECUTABLE = "python3"
EMBEDDING_MAX_ACTIVE_JOBS = 3

EMBEDDING_N_CHUNKS = {
    "TpoR": 3,
    "Ube4b": 50,
    "BRCA1": 100,
    "BF520": 30,
    "BG505": 30,
}

EMBEDDING_JOB_TIME = "06:00:00"
EMBEDDING_JOB_MEM = "20G"
MERGE_JOB_TIME = "01:00:00"
MERGE_JOB_MEM = "20G"
MERGE_POLL_SECONDS = 60
INFERENCE_JOB_TIME = "01:00:00"
INFERENCE_JOB_MEM = "16G"


## Create esmDMS Runners

Each dataset gets its own `esmDMS` object with a dataset-specific save directory.


In [5]:
runners = {}

for dataset, input_data in DATASETS.items():
    config = ESMDMSConfig(
        embedding_model="esm2_t33_650M_UR50D",
        embedding_method="per_residue",
        local_or_disk="disk",
        save_dir=str(SEQUENCE_DIR / dataset),
        dataset_name=dataset,
    )
    runners[dataset] = esmDMS(input_data=input_data, config=config)

runners


{'TpoR': <esmDMS.esmDMS at 0x7fd6280a6b00>,
 'Ube4b': <esmDMS.esmDMS at 0x7fd6280a55d0>,
 'BRCA1': <esmDMS.esmDMS at 0x7fd6280a4d00>,
 'BF520': <esmDMS.esmDMS at 0x7fd6280a5900>,
 'BG505': <esmDMS.esmDMS at 0x7fd6280a7850>}

## Process Raw Data

Raw input parsing is handled by `esmDMS.process_raw_data()`.


In [ ]:
processing_rows = []

for dataset, runner in runners.items():
    runner.process_raw_data(drop_stop_codons=True)
    df = runner.sequence_dataframe
    processing_rows.append({
        "dataset": dataset,
        "kind": DATASET_KIND[dataset],
        "rows": len(df),
        "sequence_count": df["SequenceIndex"].nunique(),
        "replicate_count": df["Replicate"].nunique(),
        "generation_count": df["Generation"].nunique(),
    })

processing_summary = pd.DataFrame(processing_rows)
processing_summary.to_csv(TABLE_DIR / "raw_processing_summary.csv", index=False)
processing_summary


## Current Class Cache Status

This checks the paths that the current `esmDMS` class will use.


In [ ]:
# TODO(esmDMS.py): add a public cache_status(layers, abstraction_method, norm_scheme) method.
cache_rows = []

for dataset, runner in runners.items():
    for layer in LAYERS:
        cache_rows.append({
            "dataset": dataset,
            "layer": layer,
            "embedding_path": str(runner._embedding_path(layer)),
            "embedding_exists": runner._embedding_path(layer).exists(),
            "inference_path": str(runner._inference_path(ABSTRACTION_METHOD, layer, NORM_SCHEME)),
            "inference_exists": runner._inference_path(ABSTRACTION_METHOD, layer, NORM_SCHEME).exists(),
        })

cache_status = pd.DataFrame(cache_rows)
cache_status.to_csv(TABLE_DIR / "current_class_cache_status.csv", index=False)
cache_status


,dataset,layer,embedding_path,embedding_exists,inference_path,inference_exists
0,TpoR,0,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False
1,TpoR,1,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False
2,TpoR,2,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False
3,TpoR,3,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False
4,TpoR,4,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False
...,...,...,...,...,...,...
165,BG505,29,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False
166,BG505,30,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False
167,BG505,31,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False
168,BG505,32,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,False


## Submit Embedding Jobs

`esmDMS.create_embedding_batch_job()` writes and submits one Slurm array job per dataset. These jobs create chunk outputs in scratch and copy them back to the dataset job directory. The array throttle keeps only `EMBEDDING_MAX_ACTIVE_JOBS` chunks active at once for each dataset.


In [ ]:
embedding_jobs = {}

SUBMIT_JOBS = False

for dataset, runner in runners.items():
    embedding_jobs[dataset] = runner.create_embedding_batch_job(
        job_dir=SEQUENCE_DIR / dataset / "embedding_batch_jobs",
        n_chunks=EMBEDDING_N_CHUNKS[dataset],
        max_active_jobs=EMBEDDING_MAX_ACTIVE_JOBS,
        job_name=f"{dataset.lower()}_esm_embed",
        partition=CLUSTER_PARTITION,
        mem=EMBEDDING_JOB_MEM,
        time=EMBEDDING_JOB_TIME,
        python_executable=PYTHON_EXECUTABLE,
        scratch_root=SCRATCH_ROOT,
        submit=SUBMIT_JOBS,
    )

pd.DataFrame(
    {
        "dataset": dataset,
        "script_path": str(job["script_path"]),
        "payload_path": str(job["payload_path"]),
        "job_id": job["job_id"],
    }
    for dataset, job in embedding_jobs.items()
)


,dataset,script_path,payload_path,job_id
0,TpoR,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,Submitted batch job 56331748
1,Ube4b,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,Submitted batch job 56331768
2,BRCA1,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,Submitted batch job 56331778
3,BF520,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,Submitted batch job 56331784
4,BG505,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,/net/dali/home/barton/dhw28/popDMS/esmDMS/data...,Submitted batch job 56331801


## Merge Completed Embedding Jobs

This cell waits for each dataset's embedding chunk files to appear, submits one Slurm merge job for that dataset, and keeps polling until every dataset has written the merged embedding cache used by inference jobs.


In [ ]:
import subprocess
import time

from IPython.display import clear_output, display

MERGE_EMBEDDING_OUTPUTS = True
SUBMIT_MERGE_JOBS = True


def slurm_job_id(job_output):
    text = str(job_output or "").strip()
    return text.split()[-1] if text else ""


def slurm_job_active(job_output):
    job_id = slurm_job_id(job_output)
    if not job_id:
        return False
    try:
        completed = subprocess.run(
            ["squeue", "-h", "-j", job_id],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        return False
    return bool(completed.stdout.strip())


def embedding_chunk_status(runner, job_dir):
    batch_dir = runner._batch_dir(job_dir)
    payload_path = runner._batch_payload_path(batch_dir)
    if not payload_path.exists():
        return batch_dir, [], [payload_path]

    payload = runner._load_pickle(payload_path)
    chunk_paths = [runner._batch_chunk_path(batch_dir, idx) for idx in range(payload["n_chunks"])]
    missing = [path for path in chunk_paths if not path.exists()]
    return batch_dir, chunk_paths, missing


def dataset_merge_complete(runner, batch_dir):
    merged_path = runner._merged_embeddings_path(batch_dir)
    layer_paths = [runner._embedding_path(layer) for layer in LAYERS]
    return merged_path.exists() and all(path.exists() for path in layer_paths)


merge_jobs = {}
merge_status = {
    dataset: {"status": "waiting_for_embedding_jobs", "missing_chunks": None, "job_id": ""}
    for dataset in runners
}

if MERGE_EMBEDDING_OUTPUTS:
    while True:
        for dataset, runner in runners.items():
            job_dir = SEQUENCE_DIR / dataset / "embedding_batch_jobs"
            batch_dir, chunk_paths, missing = embedding_chunk_status(runner, job_dir)
            embedding_job_output = embedding_jobs.get(dataset, {}).get("job_id", "") if "embedding_jobs" in globals() else ""
            embedding_active = slurm_job_active(embedding_job_output)

            if dataset_merge_complete(runner, batch_dir):
                merge_status[dataset].update({
                    "status": "merged",
                    "missing_chunks": 0,
                    "embedding_job_id": slurm_job_id(embedding_job_output),
                })
                continue

            if embedding_active or missing:
                merge_status[dataset].update({
                    "status": "waiting_for_embedding_jobs" if embedding_active else "waiting_for_chunks",
                    "missing_chunks": len(missing),
                    "embedding_job_id": slurm_job_id(embedding_job_output),
                })
                continue

            if dataset not in merge_jobs:
                merge_jobs[dataset] = runner.create_embedding_batch_merge_job(
                    job_dir=job_dir,
                    layer="all",
                    job_name=f"{dataset.lower()}_esm_merge",
                    partition=CLUSTER_PARTITION,
                    mem=MERGE_JOB_MEM,
                    time=MERGE_JOB_TIME,
                    python_executable=PYTHON_EXECUTABLE,
                    submit=SUBMIT_MERGE_JOBS,
                )

            merge_status[dataset].update({
                "status": "merge_submitted" if SUBMIT_MERGE_JOBS else "merge_scripted",
                "missing_chunks": 0,
                "embedding_job_id": slurm_job_id(embedding_job_output),
                "job_id": merge_jobs[dataset]["job_id"],
                "script_path": str(merge_jobs[dataset]["script_path"]),
            })

        clear_output(wait=True)
        display(pd.DataFrame.from_dict(merge_status, orient="index").rename_axis("dataset").reset_index())

        if all(status["status"] == "merged" for status in merge_status.values()):
            break

        time.sleep(MERGE_POLL_SECONDS)


,dataset,status,missing_chunks,job_id,embedding_job_id
0,TpoR,waiting_for_embedding_jobs,3,,56331748
1,Ube4b,waiting_for_embedding_jobs,44,,56331768
2,BRCA1,waiting_for_embedding_jobs,100,,56331778
3,BF520,waiting_for_embedding_jobs,30,,56331784
4,BG505,waiting_for_embedding_jobs,30,,56331801


## Submit Inference Jobs

`esmDMS.create_inference_job()` writes and submits one Slurm job for each dataset and layer. It reads the class-format feature cache, runs inference in scratch, and copies the result to the normal class inference cache path.


In [ ]:
inference_jobs = []

for dataset, runner in runners.items():
    for layer in LAYERS:
        job = runner.create_inference_job(
            layer=layer,
            abstraction_method=ABSTRACTION_METHOD,
            abstraction_params=ABSTRACTION_PARAMS,
            job_name=f"{dataset.lower()}_esm_infer_l{layer}",
            partition=CLUSTER_PARTITION,
            mem=INFERENCE_JOB_MEM,
            time=INFERENCE_JOB_TIME,
            python_executable=PYTHON_EXECUTABLE,
            scratch_root=SCRATCH_ROOT,
            submit=SUBMIT_JOBS,
        )
        inference_jobs.append({
            "dataset": dataset,
            "layer": layer,
            "script_path": str(job["script_path"]),
            "payload_path": str(job["payload_path"]),
            "output_path": str(job["output_path"]),
            "job_id": job["job_id"],
        })

inference_job_table = pd.DataFrame(inference_jobs)
inference_job_table.to_csv(TABLE_DIR / "submitted_inference_jobs.csv", index=False)
inference_job_table


## Load Completed Inference Results

Run this after the submitted inference jobs have completed. Loading is local, but inference itself is performed by the Slurm jobs above.


In [ ]:
LOAD_COMPLETED_INFERENCE = False
inference_results = {}

if LOAD_COMPLETED_INFERENCE:
    for dataset, runner in runners.items():
        inference_results[dataset] = {}
        for layer in LAYERS:
            inference_results[dataset][layer] = runner.load_inference_results(
                layer=layer,
                abstraction_method=ABSTRACTION_METHOD,
                norm_scheme=NORM_SCHEME,
            )


## Inference Summary

Run this after completed inference results have been loaded from the class inference cache.


In [ ]:
# TODO(esmDMS.py): add an inference_summary() method that returns this table from stored results.
inference_rows = []

for dataset, layer_results in inference_results.items():
    for layer, result in layer_results.items():
        inference_rows.append({
            "dataset": dataset,
            "kind": DATASET_KIND[dataset],
            "layer": layer,
            "n_replicates": result.s.shape[0],
            "n_dimensions": result.s.shape[1],
            "gamma_opt": result.gamma_opt,
            "s_joint_mean": result.s_joint.mean(),
            "s_joint_std": result.s_joint.std(),
        })

inference_summary = pd.DataFrame(inference_rows)
inference_summary.to_csv(TABLE_DIR / "inference_result_summary.csv", index=False)
inference_summary


## Replicate Consistency By Layer

Run this after inference jobs have completed. Layer-wise replicate consistency plots are delegated to `esmDMS.plot_avg_rep_correlations_by_layer()`.


In [ ]:
PLOT_COMPLETED_RESULTS = False

if PLOT_COMPLETED_RESULTS:
    for dataset, runner in runners.items():
        runner.plot_avg_rep_correlations_by_layer(
            layers=LAYERS,
            abstraction_method=ABSTRACTION_METHOD,
            norm_scheme=NORM_SCHEME,
            comparison="selection",
            label=dataset,
            output_path=FIGURE_DIR / f"{dataset}_selection_replicate_correlations_by_layer.png",
        )
        runner.plot_avg_rep_correlations_by_layer(
            layers=LAYERS,
            abstraction_method=ABSTRACTION_METHOD,
            norm_scheme=NORM_SCHEME,
            comparison="fitness",
            label=dataset,
            output_path=FIGURE_DIR / f"{dataset}_fitness_replicate_correlations_by_layer.png",
        )
        plt.close("all")


## Representative Replicate Scatter Plots

Replicate scatter plots are delegated to `esmDMS.plot_rep_sel_comps()` and `esmDMS.plot_rep_fit_comps()`.


In [ ]:
if PLOT_COMPLETED_RESULTS:
    for dataset, runner in runners.items():
        runner.plot_rep_sel_comps(
            layer=REPRESENTATIVE_LAYER,
            abstraction_method=ABSTRACTION_METHOD,
            norm_scheme=NORM_SCHEME,
            label=dataset,
            output_path=FIGURE_DIR / f"{dataset}_layer{REPRESENTATIVE_LAYER}_selection_replicate_scatter.png",
        )
        runner.plot_rep_fit_comps(
            layer=REPRESENTATIVE_LAYER,
            abstraction_method=ABSTRACTION_METHOD,
            norm_scheme=NORM_SCHEME,
            label=dataset,
            output_path=FIGURE_DIR / f"{dataset}_layer{REPRESENTATIVE_LAYER}_fitness_replicate_scatter.png",
        )
        plt.close("all")


## Shuffled-Frequency Control

This notebook does not implement shuffled controls locally.


In [ ]:
# TODO(esmDMS.py): add a class method for shuffled-frequency controls that shuffles
# within each (Replicate, Generation), reruns inference, and returns InferenceResult
# objects compatible with the plotting methods above.
